# Engram P2.6: Gemma as the fact writer

Gemma 3 4B writes who, what and which for 200 sentences. Benchmark data only. About 15 to 20 minutes, mostly setup.

**Kaggle:** Settings > Accelerator > GPU T4 x2, and Settings > Turn on internet. Then Save Version > Save & Run All.


In [ ]:
COMMIT = "f5ccf39"
import os, subprocess, sys, time, urllib.request
gpu = subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader", shell=True, capture_output=True, text=True).stdout.strip()
if not gpu: raise SystemExit("No GPU. Choose Settings > Accelerator > GPU T4 x2.")
print("GPU:", gpu.replace(chr(10), ", "))
try:
    urllib.request.urlopen("https://github.com", timeout=10)
except Exception:
    raise SystemExit("No internet. Choose Settings > Turn on internet, then run again.")
WORK, SAVE = "/kaggle/working", "/kaggle/working/engram_results"
os.makedirs(SAVE, exist_ok=True); os.chdir(WORK)
subprocess.run("rm -rf Engram; git clone -q https://github.com/manjunathhari/Engram.git", shell=True, check=True)
os.chdir(f"{WORK}/Engram")
subprocess.run(f"git checkout -q {COMMIT}", shell=True, check=True)
subprocess.run("apt-get -qq install -y zstd pciutils > /dev/null; curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1", shell=True)
env = dict(os.environ, OLLAMA_CONTEXT_LENGTH="4096", OLLAMA_KEEP_ALIVE="-1", OLLAMA_NUM_PARALLEL="4")
subprocess.Popen(["ollama", "serve"], env=env, stdout=open(f"{WORK}/ollama.log", "a"), stderr=subprocess.STDOUT, start_new_session=True)
for _ in range(60):
    try: urllib.request.urlopen("http://localhost:11434/api/version", timeout=2); break
    except Exception: time.sleep(2)
else: raise SystemExit("Ollama did not start. See ollama.log in the Output.")
MODEL = "hf.co/bartowski/google_gemma-3-4b-it-GGUF:Q4_K_M"
r = subprocess.run(["ollama", "pull", MODEL], capture_output=True, text=True)
if r.returncode: raise SystemExit("Model download failed: " + r.stderr[-200:])
subprocess.run(["ollama", "cp", MODEL, "gemma3:4b"])
print(subprocess.run(["ollama", "list"], capture_output=True, text=True).stdout)
LOG = f"{SAVE}/p26_run.log"
p = subprocess.run([sys.executable, "-u", "kit_tools/p26_gemma.py", "--url", "http://localhost:11434/v1", "--model", "gemma3:4b", "--save", SAVE], stdout=open(LOG, "w"), stderr=subprocess.STDOUT)
print(open(LOG).read()[-1500:])
if p.returncode or "Done" not in open(LOG).read(): raise SystemExit("The writer run did not finish. Send Claude p26_run.log.")


When it says Done, send Claude `p26_replies.json` and `p26_run.log` from the `engram_results` folder in the Output.
